# rlmflow visualization walkthrough

A guided tour of every visualization that ships with rlmflow. Everything here renders **inline in Jupyter** and runs **offline** against the saved boids trace under `examples/data/notebook-coding-agent/` — no API keys, no LLM calls.

The trace was generated by [`coding_agent.ipynb`](./coding_agent.ipynb). For the underlying `Node` query API (walk, find, where, diff, session), see [`node_basics.ipynb`](./node_basics.ipynb).

**What we cover**

1. Loading a trace
2. Inline tree rendering — `state.tree()`, Jupyter auto-render, `ascii_boxes`
3. **Interactive viewer** — `open_viewer` (Gradio: clickable graph, slider, per-agent chat)
4. Topology renders — mermaid (state / flowchart / sequence), DOT, D2
5. Step-indexed timeline — `gantt`, `gantt_html`
6. Per-node detail — `code_log`, `error_summary`, `message_stream`, `diff_system_prompts`
7. Cost & reports — `token_sparkline`, `budget_burndown`, `report_md`
8. Comparison across runs — `bench_table`
9. CLI equivalents

Full roadmap: [`docs/internal/visualization_roadmap.md`](../../docs/internal/visualization_roadmap.md).


## 1. Load a trace

`load_trace` returns a `Trace` with `states` (one snapshot per step) and `metadata`. Everything below is a function over `states`.

In [1]:
from rlmflow.utils.trace import load_trace

trace = load_trace("../data/notebook-coding-agent/trace")
states = trace.states
print(f"loaded {len(states)} states  ·  metadata: {trace.metadata or '(none)'}")

loaded 4 states  ·  metadata: {'task': 'Create a simple boids simulation in plain html and javascript. Render 100s of fast moving and coloful boids. Do not add configurations -- just the canvas. The speed should be a constant, not based on size. Split each component into separate files and delegate the work. Make sure to run and test the components after they are done. The main runnable interface should be in index.html. Save in output/boids-simulation.'}


## 2. Inline tree rendering

Every `Node` defines `_repr_html_` and `_repr_mimebundle_`, so just returning a node from a Jupyter cell renders it as a styled tree. Trees can collapse on the final `ResultNode`, so we pick the *richest* state for a complete picture.

In [2]:
richest = max(states, key=lambda s: len(s.walk()))
richest

root [supervising] {default:gpt-5}
  root.index_html [query] {fast}
  root.styles_css [query] {fast}
  root.boid_js [query] {fast}
  root.simulation_js [query] {fast}
  root.renderer_js [query] {fast}
  root.main_js [query] {fast}

In [3]:
print(richest.tree())

root [supervising] {default:gpt-5}
  root.index_html [query] {fast}
  root.styles_css [query] {fast}
  root.boid_js [query] {fast}
  root.simulation_js [query] {fast}
  root.renderer_js [query] {fast}
  root.main_js [query] {fast}


In [4]:
from rlmflow.utils.viz import ascii_boxes

print(ascii_boxes(richest))

╭──────────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.index_html  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.styles_css  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭───────────────────────────────────────── root.boid_js  [query]  {fast} ──────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭────────────────────────────────────── root.simulation_js  [query]  {fast} ───────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭─────────────────────────────────────── root.renderer_js  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
└── ╭───────────────────────────────────────── root.main_js  [query]  {fast} ──────────────────────────────────────────╮
    │                                                                                                                  │
    ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── root  [supervising]  {default:gpt-5} ────────────────────────────────────────╮
│                                                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.index_html  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
├── ╭──────────────────────────────────────── root.styles_css  [query]  {fast} ────────────────────────────────────────╮
│   │                                                                                                                  │
│   ╰───────────────────────────

## 3. Interactive viewer

The full interactive graph + per-agent chat lives in `open_viewer` (see section just below). It embeds inline in Jupyter via Gradio, with a step slider, clickable nodes, and a color-coded conversation panel for any agent you click.

For static export (PR comments, email), use the topology renders in section 4 (`mermaid`, `dot`, `d2`) or `rlmflow render <path> -f gantt-html -o gantt.html`.

### Inline Gradio viewer — clickable tree + per-agent chat

`open_viewer` ships an interactive viewer that embeds inline in Jupyter via Gradio's `inline=True` launch flag. The layout:

- **Left** — a clickable tree of every node in the run (one row per node, indented by depth, marker per type — `▢ query`, `▸ action`, `○ observation`, `↻ resume`, `• supervising`, `✓ result`, `✗ error`).
- **Right** — the **full conversation** for the agent that owns the clicked node: `system` prompt, the original `user` query, every `assistant` reply (including the REPL code blocks), every `tool` observation, and the final `done()` result.
- **Bottom** — collapsed accordion with the picked node's summary and raw JSON.

Pass `session=` (or a path to a `session/` dir) to get the *full* per-agent message history. With only `states`, you only get whatever the snapshots captured (often shallow — see how thin the conversation is for a query-only node).

The cell starts a local Gradio server in the background; call `gradio.close_all()` to stop it.

In [5]:
from rlmflow.utils.viewer import open_viewer

# Pass `session=` so the viewer can reconstruct the full per-agent chat
# (system / user / assistant / tool / result), not just the snapshot tree.
# `states` alone gives you only what was captured in the trace.
open_viewer(
    states,
    session="../data/notebook-coding-agent/session",
    inline=True,
    height=720,
    quiet=True,
)

## 4. Topology renders

Static topology renders for embedding in docs, PRs, post-mortems. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are reachable from the CLI as `rlmflow render <path> -f F`.

In [6]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(richest)}\n```")

```mermaid
stateDiagram-v2
    state "root (supervising)" as n_5d5a5eae8a6d
    [*] --> n_5d5a5eae8a6d
    n_5d5a5eae8a6d --> n_90f30a3d6c3a
    state "root.index_html (query)" as n_90f30a3d6c3a
    n_5d5a5eae8a6d --> n_d33aef65656c
    state "root.styles_css (query)" as n_d33aef65656c
    n_5d5a5eae8a6d --> n_bab57f212e77
    state "root.boid_js (query)" as n_bab57f212e77
    n_5d5a5eae8a6d --> n_7bc96c5f2af7
    state "root.simulation_js (query)" as n_7bc96c5f2af7
    n_5d5a5eae8a6d --> n_4291df9a1aee
    state "root.renderer_js (query)" as n_4291df9a1aee
    n_5d5a5eae8a6d --> n_965e03b79672
    state "root.main_js (query)" as n_965e03b79672
```

In [7]:
from rlmflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(richest)}\n```")

```mermaid
flowchart TD
    n_5d5a5eae8a6d["root<br/><i>supervising</i>"]:::sup
    n_5d5a5eae8a6d --> n_90f30a3d6c3a
    n_90f30a3d6c3a["root.index_html<br/><i>query</i>"]:::query
    n_5d5a5eae8a6d --> n_d33aef65656c
    n_d33aef65656c["root.styles_css<br/><i>query</i>"]:::query
    n_5d5a5eae8a6d --> n_bab57f212e77
    n_bab57f212e77["root.boid_js<br/><i>query</i>"]:::query
    n_5d5a5eae8a6d --> n_7bc96c5f2af7
    n_7bc96c5f2af7["root.simulation_js<br/><i>query</i>"]:::query
    n_5d5a5eae8a6d --> n_4291df9a1aee
    n_4291df9a1aee["root.renderer_js<br/><i>query</i>"]:::query
    n_5d5a5eae8a6d --> n_965e03b79672
    n_965e03b79672["root.main_js<br/><i>query</i>"]:::query
    classDef query    fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef obs      fill:#1f6feb22,stroke:#58a6ff,color:#c9d1d9;
    classDef action   fill:#d2992222,stroke:#d29922,color:#c9d1d9;
    classDef sup      fill:#bc8cff22,stroke:#bc8cff,color:#c9d1d9;
    classDef resume   fill:#7ee78722,stroke:#7ee787,color:#c9d1d9;
    classDef err      fill:#f8514922,stroke:#f85149,color:#c9d1d9;
    classDef result   fill:#3fb95022,stroke:#3fb950,color:#c9d1d9;
```

In [8]:
from rlmflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(richest)}\n```")

```mermaid
sequenceDiagram
    participant root as root
    participant root_index_html as root.index_html
    participant root_styles_css as root.styles_css
    participant root_boid_js as root.boid_js
    participant root_simulation_js as root.simulation_js
    participant root_renderer_js as root.renderer_js
    participant root_main_js as root.main_js
    root->>+root_index_html: delegate
    root->>+root_styles_css: delegate
    root->>+root_boid_js: delegate
    root->>+root_simulation_js: delegate
    root->>+root_renderer_js: delegate
    root->>+root_main_js: delegate
```

In [9]:
from rlmflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(richest))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(richest))

--- DOT (paste into a Graphviz renderer) ---
digraph rlmflow {
    rankdir=TB;
    node [shape=box, style="rounded,filled", fontname="Helvetica"];
    edge [fontname="Helvetica", fontsize=10];
    n_5d5a5eae8a6d [label="root\nsupervising", fillcolor="#bc8cff22", color="#bc8cff"];
    n_5d5a5eae8a6d -> n_90f30a3d6c3a;
    n_90f30a3d6c3a [label="root.index_html\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_5d5a5eae8a6d -> n_d33aef65656c;
    n_d33aef65656c [label="root.styles_css\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_5d5a5eae8a6d -> n_bab57f212e77;
    n_bab57f212e77 [label="root.boid_js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_5d5a5eae8a6d -> n_7bc96c5f2af7;
    n_7bc96c5f2af7 [label="root.simulation_js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_5d5a5eae8a6d -> n_4291df9a1aee;
    n_4291df9a1aee [label="root.renderer_js\nquery", fillcolor="#58a6ff22", color="#58a6ff"];
    n_5d5a5eae8a6d -> n_965e03b79672;
    n_965e03b79672 [label

## 5. Step-indexed timeline

`gantt` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html` writes a colorful self-contained HTML page; we render it inline below.

In [10]:
from rlmflow.utils.viz import gantt

gantt(states)

┏━━━━━━━━━━━━━━━━━━┳━┳━┳━┳━┓
┃agent             ┃0┃1┃2┃3┃
┡━━━━━━━━━━━━━━━━━━╇━╇━╇━╇━┩
│root              │Q│S│S│F│
│root.index_html   │ │Q│F│F│
│root.styles_css   │ │Q│F│F│
│root.boid_js      │ │Q│F│F│
│root.simulation_js│ │Q│F│F│
│root.renderer_js  │ │Q│F│F│
│root.main_js      │ │Q│F│F│
└──────────────────┴─┴─┴─┴─┘

In [11]:
from IPython.display import HTML
from rlmflow.utils.viz import gantt_html

HTML(gantt_html(states, title="boids run"))

## 6. Per-node detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent.

In [12]:
from rlmflow.utils.viz import code_log

print(code_log(states))

# [root] supervising
# Plan: delegate writing each component, then verify wiring.

contract = """
App: output/boids-simulation
Files:
- index.html: Plain page with a full-window <canvas id="boids-canvas"> and no UI. Loads ./js/main.js as type=module and links ./styles.css. Title "Boids Simulation".
- styles.css: Remove margins/scroll; black bg; canvas fills viewport and is block-level.
- js/boid.js: Exports:
    - class Boid with fields x, y, vx, vy, size, color; method update(boids, width, height, dt, speed) that applies classic boids rules (alignment, cohesion, separation) with internal constants, wraps at edges, and enforces constant speed magnitude 'speed' irrespective of size.
    - function spawnBoids(n, width, height, speed): returns array of Boid with random positions and directions; colorful hues; constant speed.
  Notes:
    - Internal constants only; no user-config UI.
    - Keep speed independent of size.
- js/simulation.js: Exports class Simulation:
    - constructor(width

In [13]:
# The boids run had no errors — synthesize a tiny example so error_summary has something to group.
from rlmflow import ErrorNode, QueryNode
from rlmflow.utils.viz import error_summary

demo = QueryNode(
    agent_id="root",
    children=[
        ErrorNode(agent_id="root.a", error="no_code_block", content="missing repl block"),
        ErrorNode(agent_id="root.b", error="no_code_block", content="empty reply"),
        ErrorNode(agent_id="root.c", error="orphaned_delegates", content="never waited"),
    ],
)
print(error_summary(demo))

3 error(s) across 2 kind(s):
  no_code_block: 2
    └─ missing repl block
  orphaned_delegates: 1
    └─ never waited


In [14]:
from pathlib import Path
from rlmflow.workspace import FileSession
from rlmflow.utils.viz import message_stream

session = FileSession(Path("../data/notebook-coding-agent/session"))
stream = message_stream("root.index_html", session)
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

(no nodes for agent 'root.index.html')


In [15]:
from rlmflow.utils.viz import diff_system_prompts

by_agent = {n.agent_id: n for n in session.load().values()}
diff = diff_system_prompts(by_agent["root.index_html"], by_agent["root.styles_css"])
print(diff[:1500] if diff != "(prompts identical)" else diff)

KeyError: 'root.index.html'

## 7. Cost & reports

Three views: a one-line ASCII sparkline of cumulative tokens, a budget burndown bar, and a full Markdown report bundling tree + cost + result + errors.

In [ ]:
from rlmflow.utils.viz import token_sparkline, budget_burndown

print(token_sparkline(states))
print(budget_burndown(states))
print(budget_burndown(states, max_budget=50_000))

In [ ]:
from rlmflow.utils.viz import report_md

Markdown(report_md(states, title="boids — coding-agent run"))

## 8. Comparison across runs

`bench_table` aggregates labeled traces — use it to compare benchmark runs side by side (cost, duration, outcome, errors).

In [ ]:
from rlmflow.utils.viz import bench_table

# Fake two runs by trimming the same trace, just to show the table shape.
print(bench_table({
    "boids:full":      states,
    "boids:firsthalf": states[: len(states) // 2 + 1],
}))

## 9. CLI equivalents

Every render here is also available without writing Python. From a shell:

```bash
rlmflow view   examples/data/notebook-coding-agent/trace
rlmflow render examples/data/notebook-coding-agent/trace -f mermaid-flowchart
rlmflow render examples/data/notebook-coding-agent/trace -f gantt-html  -o gantt.html
rlmflow render examples/data/notebook-coding-agent/trace -f report-md   -o report.md
rlmflow render examples/data/notebook-coding-agent/trace -f code-log
rlmflow render examples/data/notebook-coding-agent/trace -f tokens
```

All formats: `mermaid` · `mermaid-flowchart` · `mermaid-sequence` · `dot` · `d2` · `tree` · `ascii-boxes` · `gantt-html` · `report-md` · `code-log` · `error-summary` · `tokens`.